# Doğal Dil İşleme + Karar Bilimi (Natural Language Processing + Decision Science)

👩🏻‍🏫 Bu görevde şunları bir araya getireceğiz:
* 🗣 Doğal Dil İşleme (Natural Language Processing)
* 📊 Karar Bilimi (Decision Science)

🎯 Amaç, Olist üzerindeki ürünlerin ve satıcıların **olumsuz (kötü) yorumlarını** anlamaktır.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Data Manipulation
import numpy as np
import pandas as pd
pd.set_option("display.max_columns",None)

# Machine Learning
from sklearn.pipeline import make_pipeline

# Language Processing
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
import string
import unidecode as unidecode

# Vectorizers and NLP Models
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

🕵🏻‍♂️ Olist’in CEO’su [Tiago Dalvi](https://www.linkedin.com/in/tiagodalvi/)’nin senden yorumları okuyup anlamanı istediğini hayal et.

- Müşteriler siparişlerini **1**, **2** veya **3** puanla değerlendirdiklerinde ne söylediler?
- En sık karşılaşılan olumsuz yorumlar neler?
    - En kötü puanlanan ürünler hakkında?
    - En kötü puanlanan satıcılar hakkında?


## (0) Kurulum 🔨

Öncelikle, Olist incelemeleriyle ilgili tüm bilgileri içeren DataFrame'i yükleyeceğiz!

In [3]:
df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/olist_reviews.csv")

In [4]:
df.head()

,review_id,length_review,review_score,order_id,product_category_name,full_review
0,e64fb393e7b32834bb789ff8bb30750e,37,5,658677c97b385a9be170737859d3511b,ferramentas_jardim,Recebi bem antes do prazo estipulado.
1,f7c4243c7fe1938f181bec41a392bdeb,100,5,8e6bfb81e283fa7e4f11123a3fb894f1,esporte_lazer,Parabéns lojas lannister adorei comprar pela ...
2,8670d52e15e00043ae7de4c01cc2fe06,174,4,b9bf720beb4ab3728760088589c62129,eletroportateis,recomendo aparelho eficiente. no site a marca ...
3,4b49719c8a200003f700d3d986ea1a19,45,4,9d6f15f95d01e79bd1349cc208361f09,beleza_saude,"Mas um pouco ,travando...pelo valor ta Boa.\r\n"
4,3948b09f7c818e2d86c9a546758b2335,56,5,e51478e7e277a83743b6f9991dbfa3fb,informatica_acessorios,"Super recomendo Vendedor confiável, produto ok..."


## (2) Metin Temizleme (Text Cleaning)

🧹 `cleaning(sentence)` işlevini oluşturun ve yorumlara uygulayın. **NLTK'da Portekizce lemmatizer bulunmadığını unutmayın** (genellikle bulunmaz, ancak `nltk.stem.RSLPStemmer` kök bulucu vardır).

In [5]:
import re
import string
import unicodedata
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('rslp')

stemmer = RSLPStemmer()
stop_words = set(stopwords.words('portuguese'))

def cleaning(sentence):
    # Küçük harf
    sentence = sentence.lower()
    # Accent/unicode karakterleri normalize et (ã, ç, é vs)
    sentence = unicodedata.normalize('NFKD', sentence).encode('ascii', 'ignore').decode('utf-8')
    # Sayılar ve noktalama
    sentence = re.sub(r'[^a-z\s]', '', sentence)
    # Tokenize
    tokens = word_tokenize(sentence, language='portuguese')
    # Stopword removal + stemming
    tokens = [stemmer.stem(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

[nltk_data] Downloading package punkt to /home/ubtuna/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/ubtuna/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/ubtuna/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package rslp to /home/ubtuna/nltk_data...
[nltk_data]   Package rslp is already up-to-date!


In [7]:
df['clean_review'] = df['full_review'].astype(str).apply(cleaning)
df[['full_review', 'clean_review']].head()

,full_review,clean_review
0,Recebi bem antes do prazo estipulado.,receb bem ant praz estipul
1,Parabéns lojas lannister adorei comprar pela ...,parab loj lannist ador compr internet segur pr...
2,recomendo aparelho eficiente. no site a marca ...,recom aparelh efici sit marc aparelh impress d...
3,"Mas um pouco ,travando...pelo valor ta Boa.\r\n",pouc travandopel val boa
4,"Super recomendo Vendedor confiável, produto ok...",sup recom vend confia produt entreg ant praz


## (3) Kötü yorumların analizi

### (3.1) Düşük inceleme puanlarına sahip veri kümesi

😱 1 ile 3 arasında puan alan yorumların oranı nedir? 

In [8]:
low_score_ratio = (df['review_score'] <= 3).mean()
print(f"1-3 puan arası yorum oranı: {low_score_ratio:.2%}")

1-3 puan arası yorum oranı: 26.72%


🕵🏻‍♂️ Bu yorumlara odaklanalım...

In [9]:
bad_reviews = df[df['review_score'] <= 3].copy()
print(f"Kötü yorum sayısı: {len(bad_reviews)}")
bad_reviews.head()

Kötü yorum sayısı: 9658


,review_id,length_review,review_score,order_id,product_category_name,full_review,clean_review
14,e233e51d11511bf30e568c76360ace52,97,1,548df2c6e5f089574614894bca78acf5,eletronicos,recebi somente 1 controle Midea Split ESTILO....,receb soment control mide split estil falt con...
23,8b230a1373c6dc4bd867099fda1d7039,60,3,071251fe3b3493294536f03737a8a679,ferramentas_jardim,Eu comprei duas unidades e só recebi uma e ag...,compr dua unidad receb agor fac
24,cb2fc3e5711b5ae85e0491ee18af63ed,72,3,34e6d418f368f8079ae152bc178bc66a,beleza_saude,"Produto bom, porém o que veio para mim não co...",produt bom por vei mim nao condiz fot anunci
25,60c714ed14cef913944a3147094a4742,36,1,9ac05114800f02bfaa783bd76842dbe2,moveis_decoracao,"Produto muito inferior, mal acabado.",produt inferi mal acab
28,0bd4dcc4f6c4621baf37f73495cad8c4,16,3,a11cd01ac67beef7e8bf09740d8a35c1,esporte_lazer,Entrega no prazo,entreg praz


### (3.2) Vektörleştirme

🔡 ➡️ 🔢 Metinlerini vektörleştir.

- **Bigram**’leri (iki kelimelik ifadeler) mutlaka hesaba kat.
- Çok sık geçen kelimeleri çıkarmak için `max_df = 0.75` ayarla.
- Spoiler uyarısı: Sonunda **20.000+** kelimeye ulaşacaksın…  
  Bu challenge için sadece `max_features = 5000` ile sınırla.

In [10]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),   # unigram + bigram
    max_df=0.75,
    max_features=5000
)

X = vectorizer.fit_transform(bad_reviews['clean_review'])
print(f"Matrix shape: {X.shape}")

Matrix shape: (9658, 5000)


### (3.3) LDA

🕵🏻‍♂️ LDA'yı uyarlayın:
- `n_components = 3` seçin
- *.transform()* ile Konuların Belge Karışımını gösterin
- *.components_* ile Konu Karışımını gösterin

In [11]:
lda = LatentDirichletAllocation(
    n_components=3,
    random_state=42,
    learning_method='online'
)

lda.fit(X)
print("LDA hazır!")

LDA hazır!


#### Belge Karışımı (Konular için)

In [12]:
doc_topic_matrix = lda.transform(X)
print("Belge-Konu matrisi shape:", doc_topic_matrix.shape)
print(doc_topic_matrix[:5])

Belge-Konu matrisi shape: (9658, 3)
[[0.65034643 0.08680277 0.2628508 ]
 [0.08113212 0.43769697 0.48117091]
 [0.74085079 0.17385685 0.08529237]
 [0.79697364 0.10136544 0.10166091]
 [0.14641618 0.6584997  0.19508412]]


👉 Her inceleme için en önemli konuyu rapor edelim

In [13]:
bad_reviews['dominant_topic'] = doc_topic_matrix.argmax(axis=1)
print(bad_reviews['dominant_topic'].value_counts())

dominant_topic
1    3507
0    3458
2    2693
Name: count, dtype: int64


#### Konu Karışımı (Kelimeler için)

In [18]:
topic_word_matrix = lda.components_
print("Konu-Kelime matrisi shape:", topic_word_matrix.shape)


Konu-Kelime matrisi shape: (3, 5000)


#### Konular

🎁 Size bazı yardımcı fonksiyonlar sağladık:
- `topic_word`: Tek bir konu (topic) için en önemli kelimeleri ve ağırlıklarını döndürür
- `print_topics`: LDA tarafından bulunan farklı konuları, en önemli kelimeleriyle birlikte yazdırır

In [19]:
def topic_word(vectorizer, model, topic, topwords, with_weights = True):
    topwords_indexes = topic.argsort()[:-topwords - 1:-1]
    if with_weights == True:
        topwords = [(vectorizer.get_feature_names_out()[i], round(topic[i],2)) for i in topwords_indexes]
    if with_weights == False:
        topwords = [vectorizer.get_feature_names_out()[i] for i in topwords_indexes]
    return topwords

In [20]:
def print_topics(vectorizer, model, topwords):
    for idx, topic in enumerate(model.components_):
        print("-"*20)
        print("Topic %d:" % (idx))
        print(topic_word(vectorizer, model, topic, topwords))


🕵🏻‍♂️ Konuları en çok kullanılan kelimelerle birlikte yazdırın:

In [21]:
print_topics(vectorizer, lda, 10)

--------------------
Topic 0:
[('produt', 169.5), ('nao', 169.05), ('vei', 116.33), ('qual', 114.24), ('recom', 105.32), ('gost', 93.51), ('bem', 71.97), ('ach', 66.09), ('embal', 65.07), ('pess', 64.09)]
--------------------
Topic 1:
[('nao', 307.43), ('produt', 209.97), ('receb', 202.33), ('nao receb', 172.29), ('entreg', 155.97), ('aind', 132.12), ('receb produt', 124.13), ('compr', 96.01), ('ate', 90.46), ('aind nao', 89.01)]
--------------------
Topic 2:
[('bom', 204.3), ('compr', 179.14), ('receb', 143.89), ('entreg', 133.25), ('produt', 132.71), ('falt', 116.84), ('vei', 113.01), ('apen', 107.07), ('ped', 106.57), ('outr', 95.47)]


🇧🇷 Burada biraz Brezilya Portekizcesi kelimeler var:
- _cadeiras = chairs_
- _produto = product_
- _recomendo = recommend (não recomendo == not recommend)_
- _bom = good_
- _comprei = bought_
- _veio = came_
- _errado = wrong_
- _gostaria = I would like to..._
- _duas = two_
- _nao = not_
- _entregue = delivered_
- _pecas = part_
- _ainda = yet_
- _recebi = received_

👉 Bir konuyla ilişkili en popüler kelimeleri göster

In [25]:
# dominant_topic zaten bad_reviews'da var, df'e merge edelim
# topic_word_mixture = her topic için top kelimelerin listesi
topic_word_mixture = {
    i: topic_word(vectorizer, lda, lda.components_[i], topwords=10, with_weights=False)
    for i in range(lda.n_components)
}

# df'e most_important_topic ekle
df['most_important_topic'] = None
df.loc[bad_reviews.index, 'most_important_topic'] = bad_reviews['dominant_topic']

# most_important_words ekle
df['most_important_words'] = df['most_important_topic'].apply(
    lambda i: topic_word_mixture[i] if i is not None else None
)

df[['full_review', 'most_important_topic', 'most_important_words']].dropna().head()

,full_review,most_important_topic,most_important_words
14,recebi somente 1 controle Midea Split ESTILO....,0,"[produt, nao, vei, qual, recom, gost, bem, ach..."
23,Eu comprei duas unidades e só recebi uma e ag...,2,"[bom, compr, receb, entreg, produt, falt, vei,..."
24,"Produto bom, porém o que veio para mim não co...",0,"[produt, nao, vei, qual, recom, gost, bem, ach..."
25,"Produto muito inferior, mal acabado.",0,"[produt, nao, vei, qual, recom, gost, bem, ach..."
28,Entrega no prazo,1,"[nao, produt, receb, nao receb, entreg, aind, ..."


In [26]:
# Topic 1 (teslimat) için en popüler kelimeler
topic_idx = 1
print(f"Topic {topic_idx} en popüler kelimeler:")
print(topic_word(vectorizer, lda, lda.components_[topic_idx], topwords=15, with_weights=False))

Topic 1 en popüler kelimeler:
['nao', 'produt', 'receb', 'nao receb', 'entreg', 'aind', 'receb produt', 'compr', 'ate', 'aind nao', 'nao entreg', 'troc', 'loj', 'dia', 'correi']


In [29]:
df["most_important_words"] = df["most_important_topic"].apply(
    lambda i: topic_word_mixture[i] if i is not None and not (isinstance(i, float)) else None
)

df[['full_review', 'most_important_topic', 'most_important_words']].dropna().head()

,full_review,most_important_topic,most_important_words
14,recebi somente 1 controle Midea Split ESTILO....,0,"[produt, nao, vei, qual, recom, gost, bem, ach..."
23,Eu comprei duas unidades e só recebi uma e ag...,2,"[bom, compr, receb, entreg, produt, falt, vei,..."
24,"Produto bom, porém o que veio para mim não co...",0,"[produt, nao, vei, qual, recom, gost, bem, ach..."
25,"Produto muito inferior, mal acabado.",0,"[produt, nao, vei, qual, recom, gost, bem, ach..."
28,Entrega no prazo,1,"[nao, produt, receb, nao receb, entreg, aind, ..."


In [ ]:
df = df.rename(columns={'clean_review': 'full_review_cleaned'})

df[["review_id", "review_score", "product_category_name",
    "full_review_cleaned", "most_important_topic", "most_important_words"]].head()

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
0,e64fb393e7b32834bb789ff8bb30750e,5,ferramentas_jardim,receb bem ant praz estipul,None,None
1,f7c4243c7fe1938f181bec41a392bdeb,5,esporte_lazer,parab loj lannist ador compr internet segur pr...,None,None
2,8670d52e15e00043ae7de4c01cc2fe06,4,eletroportateis,recom aparelh efici sit marc aparelh impress d...,None,None
3,4b49719c8a200003f700d3d986ea1a19,4,beleza_saude,pouc travandopel val boa,None,None
4,3948b09f7c818e2d86c9a546758b2335,5,informatica_acessorios,sup recom vend confia produt entreg ant praz,None,None


In [32]:
df[["review_id",
        "review_score",
        "product_category_name",
        "full_review_cleaned",
        "most_important_topic",
        "most_important_words"]
      ].head()

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
0,e64fb393e7b32834bb789ff8bb30750e,5,ferramentas_jardim,receb bem ant praz estipul,None,None
1,f7c4243c7fe1938f181bec41a392bdeb,5,esporte_lazer,parab loj lannist ador compr internet segur pr...,None,None
2,8670d52e15e00043ae7de4c01cc2fe06,4,eletroportateis,recom aparelh efici sit marc aparelh impress d...,None,None
3,4b49719c8a200003f700d3d986ea1a19,4,beleza_saude,pouc travandopel val boa,None,None
4,3948b09f7c818e2d86c9a546758b2335,5,informatica_acessorios,sup recom vend confia produt entreg ant praz,None,None


## (3.4) Pipeline Tf-Idf ve LDA

In [33]:
from sklearn import set_config
set_config("diagram")

🔨 Önceki Vectorizer ve LDA'yı birbirine bağlayan bir Pipeline oluşturun.

Temizlenmiş metinlere uyarlayın.

In [35]:
pipeline = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), max_df=0.75, max_features=5000),
    LatentDirichletAllocation(n_components=3, random_state=42, learning_method='online')
)

bad_reviews = bad_reviews.rename(columns={'clean_review': 'full_review_cleaned'})

pipeline.fit(bad_reviews['full_review_cleaned'])
print("Pipeline hazır!")

Pipeline hazır!


💡 `pipeline.components_` ile bileşenlere erişmeye çalışırsanız, Pipeline'da `components_` olmadığı için bu YÜRÜMEZ. Ancak, LDA'ya erişmek için `pipeline._final_estimator` kullanabilirsiniz. Ve buradan konulara erişebilirsiniz!

In [36]:
pipeline._final_estimator

LatentDirichletAllocation(learning_method='online', n_components=3,
                          random_state=42)

In [37]:
pipeline._final_estimator.components_

array([[ 3.80052021, 12.66321988,  2.15714195, ...,  0.5011188 ,
         0.35693567,  6.99219328],
       [ 0.60851702,  0.39946224,  0.91525275, ...,  2.85018708,
         8.16025398,  0.34205347],
       [ 0.3857827 ,  0.35095705,  0.45964902, ...,  0.35038789,
         0.66567334,  0.34046622]])

Pipeline ile **Belge Karışımı**:

In [38]:
doc_topic_pipeline = pipeline.transform(bad_reviews['full_review_cleaned'])
print("Belge-Konu matrisi shape:", doc_topic_pipeline.shape)
print(doc_topic_pipeline[:3])

Belge-Konu matrisi shape: (9658, 3)
[[0.65034643 0.08680277 0.2628508 ]
 [0.08113212 0.43769697 0.48117091]
 [0.74085079 0.17385685 0.08529237]]


Pipeline ile **Konu Karışımı**:

In [39]:
print_topics(pipeline[0], pipeline._final_estimator, 10)

--------------------
Topic 0:
[('produt', 169.5), ('nao', 169.05), ('vei', 116.33), ('qual', 114.24), ('recom', 105.32), ('gost', 93.51), ('bem', 71.97), ('ach', 66.09), ('embal', 65.07), ('pess', 64.09)]
--------------------
Topic 1:
[('nao', 307.43), ('produt', 209.97), ('receb', 202.33), ('nao receb', 172.29), ('entreg', 155.97), ('aind', 132.12), ('receb produt', 124.13), ('compr', 96.01), ('ate', 90.46), ('aind nao', 89.01)]
--------------------
Topic 2:
[('bom', 204.3), ('compr', 179.14), ('receb', 143.89), ('entreg', 133.25), ('produt', 132.71), ('falt', 116.84), ('vei', 113.01), ('apen', 107.07), ('ped', 106.57), ('outr', 95.47)]


## (4) 🎁 Ürün Kategorileri

### (4.1) Ürün kategorilerine göre gruplandırma

📈 Veri kümesini `product_category_name` ile gruplandırın ve performanslarını inceleyin.

In [40]:
# Performansa göre ürün kategorileri - sayı, ortalama, medyan ve standart sapmaya bakın
product_categories = df.groupby(by = 'product_category_name').agg({
        'review_score': ["count", "mean", "median", "std"]
    })

# Analiz için belirli bir süreden daha az satılan ürünleri kaldırma
cutoff = 50
product_categories = product_categories[product_categories[("review_score", "count")] > cutoff]

# Ürün kategorilerini performansa göre sıralama
product_categories = product_categories.sort_values(by = [('review_score', 'mean'),
                                                          ('review_score', 'std')],
                                                    ascending = [False, True])
product_categories

review_score                   \
                                                      count      mean median   
product_category_name                                                          
alimentos_bebidas                                        75  4.560000    5.0   
livros_interesse_geral                                  152  4.539474    5.0   
malas_acessorios                                        375  4.317333    5.0   
livros_tecnicos                                          83  4.301205    5.0   
pcs                                                      67  4.283582    5.0   
alimentos                                               151  4.231788    5.0   
papelaria                                               758  4.201847    5.0   
instrumentos_musicais                                   228  4.192982    5.0   
fashion_calcados                                        101  4.188119    5.0   
fashion_bolsas_e_acessorios                             726  4.168044    5.0   
perfumaria                                             1094  4.167276    5.0   
construcao_ferramentas_jardim                            85  4.164706    5.0   
cool_stuff                                             1360  4.154412    5.0   
ferramentas_jardim                                     1391  4.105679    5.0   
beleza_saude                                           3046  4.104071    5.0   
brinquedos                                             1271  4.101495    5.0   
esporte_lazer                                          2637  4.097838    5.0   
bebidas                                                 111  4.090090    5.0   
pet_shop                                                603  4.086235    5.0   
construcao_ferramentas_iluminacao                        86  4.081395    5.0   
eletroportateis                                         242  4.061983    5.0   
automotivo                                             1533  4.049576    5.0   
eletrodomesticos                                        354  4.033898    5.0   
eletrodomesticos_2                                       89  4.011236    5.0   
relogios_presentes                                     2211  4.009950    5.0   
construcao_ferramentas_construcao                       277  4.007220    5.0   
consoles_games                                          339  4.005900    5.0   
utilidades_domesticas                                  2192  4.005018    5.0   
eletronicos                                             942  3.979830    5.0   
industria_comercio_e_negocios                            82  3.963415    5.0   
bebes                                                   882  3.955782    5.0   
sinalizacao_e_seguranca                                  61  3.934426    5.0   
telefonia                                              1654  3.874244    5.0   
cama_mesa_banho                                        3891  3.853765    5.0   
artes                                                    80  3.850000    5.0   
casa_conforto                                           175  3.817143    5.0   
moveis_decoracao                                       2311  3.815232    5.0   
moveis_sala                                             138  3.789855    5.0   
informatica_acessorios                                 2473  3.774767    5.0   
audio                                                   134  3.768657    5.0   
market_place                                            102  3.745098    4.0   
moveis_cozinha_area_de_servico_jantar_e_jardim           94  3.734043    4.0   
construcao_ferramentas_seguranca                         61  3.721311    5.0   
agro_industria_e_comercio                                57  3.701754    4.0   
telefonia_fixa                                           96  3.697917    4.0   
casa_construcao                                         177  3.672316    4.0   
climatizacao                                             93  3.655914    4.0   
fashion_roupa_masculina                        

### (4.2) En kötü ürün kategorileri

👎 *Ortalama değerlendirme puanı* açısından en kötü beş kategoriyi `worst_products` adlı bir değişkene kaydedin.

In [41]:
worst_products = product_categories.tail(5).sort_values(by = [("review_score", "count")],
                                                       ascending = False)
worst_products

review_score                           
                               count      mean median       std
product_category_name                                          
moveis_escritorio                546  3.298535    4.0  1.562366
casa_construcao                  177  3.672316    4.0  1.498094
telefonia_fixa                    96  3.697917    4.0  1.576938
climatizacao                      93  3.655914    4.0  1.625173
fashion_roupa_masculina           56  3.482143    4.0  1.737198

👇 Yalnızca `worst_products` öğelerini içeren bir `worst_products_review` DataFrame oluşturun.

In [42]:
worst_products_reviews = df[df.product_category_name.isin(worst_products.index)]
worst_products_reviews[["review_id",
                        "review_score",
                        "product_category_name",
                        "full_review_cleaned",
                        "most_important_topic",
                        "most_important_words"]
      ]

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
45,c422274e50e900b46fee429016c5458d,3,moveis_escritorio,gost,0,"[produt, nao, vei, qual, recom, gost, bem, ach..."
126,5d6f9cddc8335878d8bf20c3bd4602e8,5,moveis_escritorio,meg recom receb produt corret dentr prazoverif...,None,None
212,1f836d160f50247a394034abd93b7c24,3,casa_construcao,esper instal,0,"[produt, nao, vei, qual, recom, gost, bem, ach..."
219,edd86e0af4dc62b340c60d846643cbf9,5,moveis_escritorio,produt praz estendidopor entreg praz divulg pr...,None,None
232,e47020cf52099bce9dc57524ea41218b,4,moveis_escritorio,indic loj compr sempr lannist,None,None
...,...,...,...,...,...,...
35956,123381026f13c723cb967b14b1bc673a,5,moveis_escritorio,ador cade bonit conforta facil mont mesm mont ...,None,None
36031,042b7c995635364a03c5bb79e994f513,4,fashion_roupa_masculina,filh gost,None,None
36034,5d4beea8e2c71baee7f55c7ae3ae3313,1,moveis_escritorio,insatisfeit cade vei defeit fabr solt encaix p...,0,"[produt, nao, vei, qual, recom, gost, bem, ach..."
36037,0cac8fef1bcba2a3335b132807520f6d,1,moveis_escritorio,produt vei fur err mont infeliz cade nao vai d...,0,"[produt, nao, vei, qual, recom, gost, bem, ach..."


### (4.3). En kötü ürünler için konular

❓ En kötü ürünlerin konuları nelerdir? ❓

In [43]:
worst_products_reviews["most_important_topic"].value_counts()

most_important_topic
0    162
1    132
2    120
Name: count, dtype: int64

In [44]:
bad_frequency = list(worst_products_reviews["most_important_topic"].value_counts().index)
bad_frequency

[0, 1, 2]

In [45]:
[topic_word_mixture[i] for i in bad_frequency]

[['produt',
  'nao',
  'vei',
  'qual',
  'recom',
  'gost',
  'bem',
  'ach',
  'embal',
  'pess'],
 ['nao',
  'produt',
  'receb',
  'nao receb',
  'entreg',
  'aind',
  'receb produt',
  'compr',
  'ate',
  'aind nao'],
 ['bom',
  'compr',
  'receb',
  'entreg',
  'produt',
  'falt',
  'vei',
  'apen',
  'ped',
  'outr']]

## (5) 🎁 Satıcılar...

* En kötü satıcılar tarafından ne tür ürünler satıldı?
* En kötü satıcılar için başlıca yorumlar nelerdir?

### (5.1) En kötü satıcılar

In [48]:
import sys
sys.path.insert(0, '/home/ubtuna')

from olist.seller import Seller
sellers = Seller().get_training_data()
sellers.columns

Olist class initialized!


Index(['seller_id', 'n_orders', 'quantity', 'sales', 'quantity_per_order',
       'seller_city', 'seller_state', 'delay_to_carrier', 'wait_time',
       'date_first_sale', 'date_last_sale', 'months_on_olist',
       'share_of_five_stars', 'share_of_one_stars', 'review_score', 'revenues',
       'cost_of_reviews', 'profits'],
      dtype='object')

In [49]:
from olist.seller import Seller
sellers = Seller().get_training_data()
sellers.columns

Olist class initialized!


Index(['seller_id', 'n_orders', 'quantity', 'sales', 'quantity_per_order',
       'seller_city', 'seller_state', 'delay_to_carrier', 'wait_time',
       'date_first_sale', 'date_last_sale', 'months_on_olist',
       'share_of_five_stars', 'share_of_one_stars', 'review_score', 'revenues',
       'cost_of_reviews', 'profits'],
      dtype='object')

👇 En kötü satan 10 ürünü seçin ve bunları `worst_sellers` adlı bir değişkene kaydedin.

In [50]:
worst_sellers = sellers[["seller_id", "review_score", "profits"]].sort_values(
    by = "profits",
    ascending = True).head(10)
worst_sellers

,seller_id,review_score,profits
1189,6560211a19b47992c3666cc44a7e94c0,3.909406,-26583.052402
358,1f50f920176fa81dab994f9023523100,3.982402,-25486.916346
1479,7c67e1448b00f6e969d365cea6b010ab,3.348208,-24166.914733
857,4a3ca9315b744ce9f8e9374361493884,3.803931,-23365.298458
2385,cc419e0650a3c5ba77189a1882b7556a,4.069575,-19455.106043
2722,ea8482cd71df3c1969d7b9473ff13abc,3.953216,-17907.798723
188,1025f0e2d44d7041d6cf58b6550e0bfa,3.849755,-16383.935454
1642,8b321bb669392f5163d04c59e235e066,3.995069,-15950.395146
2447,d2374cbcbb3ca4ab1086534108cc3ab7,3.636364,-13923.987161
277,1835b56ce799e6a4dc4eddc053f04066,3.593863,-11489.824591


### (5.2) En kötü satıcılar tarafından satılan ürünler

In [56]:
import os
print(os.listdir('/home/ubtuna/olist/'))

['README.md', '.direnv', 'seller.py', 'order.py', '.envrc.CALISIYOR-DOKUNMA', '__init__.py', '.vscode', 'product.py', '__pycache__', '.envrc', 'data.py', 'utils.py', 'review.py']


In [57]:
import pandas as pd

products = pd.read_csv('/home/ubtuna/.workintech/olist/data/csv/olist_products_dataset.csv')[['product_id', 'product_category_name']].rename(columns={'product_category_name': 'category'})
products

,product_id,category
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria
1,3aa071139cb16b67ca9e5dea641aaa2f,artes
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer
3,cef67bcfe19066a932b7673e239eb23d,bebes
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas
...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios


In [61]:
# data["order_items"] yerine direkt CSV'den çekelim
order_items = pd.read_csv('/home/ubtuna/.workintech/olist/data/csv/olist_order_items_dataset.csv')

sellers_product_category = order_items.merge(products, on="product_id", how="left")[["seller_id", "category"]]
sellers_product_category

,seller_id,category
0,48436dade18ac8b2bce089ec2a041202,cool_stuff
1,dd7ddc04e1b6c2c614352b383efe2d36,pet_shop
2,5b51032eddd242adc84c38acab88f23d,moveis_decoracao
3,9d7a1d34a5052409006425275ba1c2b4,perfumaria
4,df560393f3a51e74553ab94004ba5c87,ferramentas_jardim
...,...,...
112645,b8bc237ba3788b23da09c0f1f3a3288c,utilidades_domesticas
112646,f3c38ab652836d21de61fb8314b69182,informatica_acessorios
112647,c3cfdc648177fdbbbb35635a37472c53,esporte_lazer
112648,2b3e4a2a3ea8e01938cabda2a3e5cc79,informatica_acessorios


❓ En kötü satıcılar tarafından satılan ürün türleri nelerdir? ❓

In [64]:
sellers_product_category = order_items.merge(products, on="product_id", how="left")[["seller_id", "category"]]
sellers_product_category

,seller_id,category
0,48436dade18ac8b2bce089ec2a041202,cool_stuff
1,dd7ddc04e1b6c2c614352b383efe2d36,pet_shop
2,5b51032eddd242adc84c38acab88f23d,moveis_decoracao
3,9d7a1d34a5052409006425275ba1c2b4,perfumaria
4,df560393f3a51e74553ab94004ba5c87,ferramentas_jardim
...,...,...
112645,b8bc237ba3788b23da09c0f1f3a3288c,utilidades_domesticas
112646,f3c38ab652836d21de61fb8314b69182,informatica_acessorios
112647,c3cfdc648177fdbbbb35635a37472c53,esporte_lazer
112648,2b3e4a2a3ea8e01938cabda2a3e5cc79,informatica_acessorios


In [65]:
sellers_product_category.groupby("seller_id").count()

,category
seller_id,
0015a82c2db000af6aaaf3ae2ecb0532,3
001cca7ae9ae17fb1caed9dfb1094831,239
001e6ad469a905060d959994f1b41e4f,1
002100f778ceb8431b7a1020ff7ab48f,55
003554e2dce176b5555353e4f3555ac8,0
...,...
ffcfefa19b08742c5d315f2791395ee5,1
ffdd9f82b9a447f6f8d4b91554cc7dd3,20
ffeee66ac5d5a62fe688b9d26f83f534,14


### (5.3) En kötü satıcılar için kategoriler ve konular

🎁 İşte bazı kullanışlı işlevler:
- Bir satıcı tarafından satılan ürün kategorilerini göstermek için `focus_seller(seller_id)`
- Bir satıcı için en sık kullanılan konuların en popüler kelimelerini göstermek için `bad_reviews_seller`

In [66]:
def focus_seller(seller_id):
    return sellers_product_category[sellers_product_category.seller_id == seller_id].value_counts()

In [ ]:
# worst_products_reviews = bad_reviews + order_items merge
worst_products_reviews = bad_reviews.merge(
    order_items[["order_id", "seller_id", "product_id"]],
    on="order_id",
    how="left"
)

# bad_reviews_sellers
bad_reviews_sellers = worst_products_reviews.merge(
    order_items[["order_id", "seller_id"]],
    on="order_id",
    how="left"
)

bad_reviews_sellers.head(3)

,review_id,length_review,review_score,order_id,product_category_name,full_review,full_review_cleaned,dominant_topic,seller_id_x,product_id,seller_id_y
0,e233e51d11511bf30e568c76360ace52,97,1,548df2c6e5f089574614894bca78acf5,eletronicos,recebi somente 1 controle Midea Split ESTILO....,receb soment control mide split estil falt con...,0,128639473a139ac0f3e5f5ade55873a5,62ad9a8972411e333e16347051a98e2a,128639473a139ac0f3e5f5ade55873a5
1,e233e51d11511bf30e568c76360ace52,97,1,548df2c6e5f089574614894bca78acf5,eletronicos,recebi somente 1 controle Midea Split ESTILO....,receb soment control mide split estil falt con...,0,128639473a139ac0f3e5f5ade55873a5,62ad9a8972411e333e16347051a98e2a,8b321bb669392f5163d04c59e235e066
2,e233e51d11511bf30e568c76360ace52,97,1,548df2c6e5f089574614894bca78acf5,eletronicos,recebi somente 1 controle Midea Split ESTILO....,receb soment control mide split estil falt con...,0,8b321bb669392f5163d04c59e235e066,566a4f2c4385f36d15c00dfcaae132d1,128639473a139ac0f3e5f5ade55873a5


In [70]:
def bad_reviews_seller(bad_reviews_sellers, seller_id):
    mask = (bad_reviews_sellers.seller_id == seller_id)
    temp = bad_reviews_sellers[mask]
    if len(temp) > 0: # satıcı kötü yorumlar veri çerçevesinde görünüyorsa
        most_frequent_topic_seller = list(temp.most_important_topic.value_counts().head(1).index)[0]
        return topic_word_mixture[most_frequent_topic_seller]

❓Bu en az satan ürünlerin her biri için en sık kullanılan ürün kategorilerini ve kelimeleri gösterin ❓

In [72]:
print(bad_reviews_sellers.columns.tolist())

['review_id', 'length_review', 'review_score', 'order_id', 'product_category_name', 'full_review', 'full_review_cleaned', 'dominant_topic', 'seller_id_x', 'product_id', 'seller_id_y']


In [73]:
bad_reviews_sellers['seller_id'] = bad_reviews_sellers['seller_id_x'].fillna(bad_reviews_sellers['seller_id_y'])
bad_reviews_sellers['most_important_topic'] = bad_reviews_sellers['dominant_topic']


🏁 Tebrikler. NLP'nin bazı temellerini (Ön İşleme + Vektörleştirme + NB/LDA) öğrendiniz ve bu yeni “uzmanlığı” Karar Bilimi ile birleştirdik.

💾 `git add / commit / push` yapmayı unutmayın.